# Inference using RL post-trained Alpamayo model (VLM backbone with discrete action-token variant)

In [ ]:
import sys

sys.path[:0] = ["../../../src", "../.."]

MODEL_PATH = "/path/to/exported_model/"
PAI_DIR = "/path/to/PAI_mini"
CLIP_INDEX = "clip_index_mini.parquet"

In [ ]:
import copy
import numpy as np
import torch
from rl.models.reasoning_vla.base_model import RLWrapperReasoningVLA
from alpamayo1_5.data.pai_utils import PhysicalAIAVDatasetLocalInterface
from alpamayo1_5.helper import create_message, get_processor, to_device
from alpamayo1_5.load_physical_aiavdataset import load_physical_aiavdataset

### Load model and construct data preprocessor

In [ ]:
model = RLWrapperReasoningVLA.from_pretrained(MODEL_PATH, torch_dtype=torch.bfloat16).cuda().eval()
processor = get_processor(model.tokenizer)

### Load and prepare data

In [ ]:
avdi = PhysicalAIAVDatasetLocalInterface(local_dir=PAI_DIR, clip_index_metadata=CLIP_INDEX)
clip_id = avdi.get_all_clip_ids()[0]
data = load_physical_aiavdataset(clip_id, avdi=avdi)

messages = create_message(data["image_frames"].flatten(0, 1))
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False,
    continue_final_message=True,
    return_dict=True,
    return_tensors="pt",
)
print("seq length:", inputs.input_ids.shape)

model_inputs = to_device(
    {
        "tokenized_data": inputs,
        "ego_history_xyz": data["ego_history_xyz"],
        "ego_history_rot": data["ego_history_rot"],
    },
    "cuda",
)

### Model inference

In [ ]:
with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
    pred_xyz, pred_rot, _, extra = model.sample_trajectories_from_data(
        data=copy.deepcopy(model_inputs),
        temperature=0.6,
        top_p=0.98,
        num_traj_samples=6,
        max_generation_length=256,
        return_extra=True,
    )

print("Chain-of-Causation (per trajectory):\n", extra["cot"][0])

## Visualizing data and results

In [ ]:
import matplotlib.pyplot as plt

gt_xy = data["ego_future_xyz"].cpu()[0, 0, :, :2].T.numpy()
for i in range(pred_xyz.shape[2]):
    pred_xy = pred_xyz.cpu()[0, 0, i, :, :2].T.numpy()
    plt.plot(pred_xy[0], pred_xy[1], "o-", label=f"Predicted Trajectory #{i + 1}")
plt.plot(gt_xy[0], gt_xy[1], "r-", label="Ground Truth Trajectory")
plt.xlabel("x (m)")
plt.ylabel("y (m)")
plt.legend(loc="best")
plt.axis("equal")
plt.show()

pred_xy_all = pred_xyz.cpu().numpy()[0, 0, :, :, :2].transpose(0, 2, 1)
ades = np.linalg.norm(pred_xy_all - gt_xy[None, ...], axis=1).mean(-1)
print(f"minADE: {ades.min():.3f} m  meanADE: {ades.mean():.3f} m")